# Multi-Modal Stock Price Movement Classification
## ViT-TFT Hybrid: Vision + Temporal Fusion — Demo Notebook

**Paper:** *A Multi-Modal Approach Using a Hybrid Vision Transformer and Temporal Fusion Transformer Model for Stock Price Movement Classification* (Friday et al., IEEE Access 2025)

This notebook runs the **complete end-to-end pipeline**:
1. Data download (yfinance)
2. Technical-indicator feature engineering
3. Candlestick image generation + HOG extraction
4. Multi-modal model training (CNN + LSTM-Attention late fusion)
5. LSTM-only baseline training
6. Classification evaluation (accuracy / F1 / MCC / ROC)
7. Quant backtesting (ROI / Sharpe / Max Drawdown)
8. Statistical validation (paired t-test)
9. All result visualisations


## 0 · Setup

In [ ]:
# ── Install dependencies (uncomment on first run / Colab) ─────────────────
# !pip install -q yfinance ta mplfinance scikit-image torch torchvision seaborn tqdm PyYAML

import sys, os
# Make sure the project root is on the path
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams["figure.dpi"] = 110

import torch
print(f"PyTorch: {torch.__version__}")

from src.utils import load_config, set_seed, get_device, ensure_dir

set_seed(42)
device = get_device()
cfg    = load_config("../configs/config.yaml")
print(f"Device : {device}")
print(f"Config : {cfg['data']['tickers']}  |  {cfg['data']['start_date']} → {cfg['data']['end_date']}")

## 1 · Data Download

In [ ]:
from src.data_loader import download_all, summarise

ensure_dir("../data/cache")
ensure_dir("../data/raw")

dfs = download_all(
    tickers   = cfg["data"]["tickers"],
    start     = cfg["data"]["start_date"],
    end       = cfg["data"]["end_date"],
    cache_dir = "../data/cache",
)

for ticker, df in dfs.items():
    summarise(df, ticker)

# Pick primary ticker for demo
TICKER = list(dfs.keys())[0]
df_main = dfs[TICKER].copy()
print(f"\nPrimary ticker: {TICKER}  |  {len(df_main)} rows")
df_main.head()

## 2 · Feature Engineering

In [ ]:
from src.feature_engineering import (
    add_technical_indicators, get_feature_matrix,
    build_sequences, FEATURE_COLS
)
from src.data_loader import label_price_movement

HORIZON = 1          # 1-day ahead forecast
WINDOW  = cfg["images"]["window_sizes"][0]   # e.g. 5

df_feat = add_technical_indicators(df_main)
labels_series = label_price_movement(
    df_feat,
    n=HORIZON,
    threshold=cfg["labelling"]["threshold"]
)
df_feat["label"] = labels_series

# Drop NaN rows (from indicator warm-up + label shift)
df_feat.dropna(inplace=True)

print(f"Feature matrix shape: {df_feat.shape}")
print(f"Label distribution:\n{df_feat['label'].value_counts().sort_index()}")

In [ ]:
feat_matrix = get_feature_matrix(df_feat, normalise=True)
label_arr   = df_feat["label"].values.astype(np.float32)

X_seq, y_seq = build_sequences(feat_matrix, label_arr, window=WINDOW)
print(f"Sequences X: {X_seq.shape}  y: {y_seq.shape}")

## 3 · Candlestick Image Generation + HOG Features

In [ ]:
from src.image_generator import generate_images_for_ticker, images_to_tensor_array
from src.hog_features import extract_hog_batch

IMG_SIZE = cfg["images"]["img_size"]

# Build parallel list of window DataFrames (needed for candle attributes)
window_dfs = [df_feat.iloc[i : i + WINDOW] for i in range(len(df_feat) - WINDOW + 1)]

images, img_indices = generate_images_for_ticker(
    df_feat,
    window=WINDOW,
    img_size=IMG_SIZE,
    dpi=cfg["images"]["dpi"],
    style=cfg["images"]["style"],
    cache_dir="../data/cache",
    ticker=TICKER,
)
print(f"Generated {len(images)} images  (shape of first: {images[0].shape})")

# Preview
fig, axes = plt.subplots(1, 5, figsize=(12, 2.5))
for ax, idx in zip(axes, np.linspace(0, len(images)-1, 5, dtype=int)):
    ax.imshow(images[idx])
    ax.axis("off")
    ax.set_title(f"t={idx}")
fig.suptitle(f"{TICKER} — Sample candlestick charts (window={WINDOW})", fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
hog_cfg = cfg["hog"]
hog_arr = extract_hog_batch(
    images,
    window_dfs=window_dfs,
    orientations=hog_cfg["orientations"],
    pixels_per_cell=tuple(hog_cfg["pixels_per_cell"]),
    cells_per_block=tuple(hog_cfg["cells_per_block"]),
    include_candle_feats=True,
)
img_tensor_arr = images_to_tensor_array(images, img_size=IMG_SIZE)

print(f"HOG feature matrix: {hog_arr.shape}")
print(f"Image tensor array: {img_tensor_arr.shape}")

## 4 · Train/Test Split (Time-Series)

In [ ]:
# Align all arrays (X_seq, img_tensor_arr, hog_arr share the same index)
N = min(len(X_seq), len(img_tensor_arr), len(hog_arr))
X_seq        = X_seq[:N]
y_seq        = y_seq[:N]
img_arr      = img_tensor_arr[:N]
hog_arr_     = hog_arr[:N]

# 80 / 20 time-ordered split
split  = int(0.80 * N)
tr_idx = slice(None, split)
te_idx = slice(split, None)

print(f"Train: {split}  Test: {N - split}")

## 5 · Multi-Modal Model Training

In [ ]:
from src.train import StockDataset, build_loaders, train_model
from src.model import build_multimodal_model, build_baseline_model

ensure_dir("../results/checkpoints")

train_ds_mm = StockDataset(
    sequences = X_seq[tr_idx],
    labels    = y_seq[tr_idx].astype(np.int64),
    images    = img_arr[tr_idx],
    hog_feats = hog_arr_[tr_idx],
)
test_ds_mm = StockDataset(
    sequences = X_seq[te_idx],
    labels    = y_seq[te_idx].astype(np.int64),
    images    = img_arr[te_idx],
    hog_feats = hog_arr_[te_idx],
)

train_loader_mm, val_loader_mm = build_loaders(
    train_ds_mm, test_ds_mm,
    batch_size=cfg["training"]["batch_size"],
    balance=True,
)

mm_model = build_multimodal_model(
    num_ts_features  = X_seq.shape[-1],
    num_hog_features = hog_arr_.shape[-1],
    cfg              = cfg,
)
print(f"Multi-modal params: {sum(p.numel() for p in mm_model.parameters()):,}")

hist_mm = train_model(
    mm_model,
    train_loader_mm,
    val_loader_mm,
    cfg=cfg,
    is_multimodal=True,
    ckpt_path=f"../results/checkpoints/{TICKER}_multimodal_best.pt",
)

## 6 · Baseline LSTM Training

In [ ]:
train_ds_bl = StockDataset(
    sequences = X_seq[tr_idx],
    labels    = y_seq[tr_idx].astype(np.int64),
)
test_ds_bl = StockDataset(
    sequences = X_seq[te_idx],
    labels    = y_seq[te_idx].astype(np.int64),
)
train_loader_bl, val_loader_bl = build_loaders(
    train_ds_bl, test_ds_bl,
    batch_size=cfg["training"]["batch_size"],
    balance=True,
)

bl_model = build_baseline_model(num_ts_features=X_seq.shape[-1], cfg=cfg)
print(f"Baseline params: {sum(p.numel() for p in bl_model.parameters()):,}")

hist_bl = train_model(
    bl_model,
    train_loader_bl,
    val_loader_bl,
    cfg=cfg,
    is_multimodal=False,
    ckpt_path=f"../results/checkpoints/{TICKER}_baseline_best.pt",
)

## 7 · Classification Evaluation

In [ ]:
from src.train import predict
from src.evaluate import (
    compute_metrics, print_metrics, full_report,
    plot_confusion_matrix, plot_roc_curve,
    plot_training_history, benchmark_comparison
)

ensure_dir("../results")

# Multi-modal predictions
mm_preds, mm_probs = predict(mm_model, val_loader_mm, device, is_multimodal=True)
bl_preds, bl_probs = predict(bl_model, val_loader_bl, device, is_multimodal=False)
y_test = y_seq[te_idx].astype(np.int64)

mm_metrics = compute_metrics(y_test, mm_preds, mm_probs)
bl_metrics = compute_metrics(y_test, bl_preds, bl_probs)

print_metrics(mm_metrics, "Multi-Modal")
print_metrics(bl_metrics, "Baseline LSTM")

In [ ]:
print("── Multi-Modal Classification Report ──")
print(full_report(y_test, mm_preds))
print("── Baseline LSTM Classification Report ──")
print(full_report(y_test, bl_preds))

In [ ]:
# Training curves
_ = plot_training_history(hist_mm, title=f"{TICKER} Multi-Modal — Training History",
                           save_path=f"../results/{TICKER}_mm_training.png")
plt.show()
_ = plot_training_history(hist_bl, title=f"{TICKER} Baseline — Training History",
                           save_path=f"../results/{TICKER}_bl_training.png")
plt.show()

In [ ]:
# Confusion matrices
_ = plot_confusion_matrix(y_test, mm_preds,
                           title=f"{TICKER} Multi-Modal — Confusion Matrix",
                           save_path=f"../results/{TICKER}_mm_confusion.png")
plt.show()
_ = plot_confusion_matrix(y_test, bl_preds,
                           title=f"{TICKER} Baseline — Confusion Matrix",
                           save_path=f"../results/{TICKER}_bl_confusion.png")
plt.show()

In [ ]:
# ROC curves
_ = plot_roc_curve(y_test, mm_probs,
                   title=f"{TICKER} Multi-Modal — ROC Curve",
                   save_path=f"../results/{TICKER}_mm_roc.png")
plt.show()
_ = plot_roc_curve(y_test, bl_probs,
                   title=f"{TICKER} Baseline — ROC Curve",
                   save_path=f"../results/{TICKER}_bl_roc.png")
plt.show()

In [ ]:
# Benchmark comparison bar chart
_ = benchmark_comparison(bl_metrics, mm_metrics,
                          save_path=f"../results/{TICKER}_benchmark.png")
plt.show()

## 8 · Quant Backtesting

In [ ]:
from src.backtest import run_full_backtest

# Align test prices with test predictions
test_dates  = df_feat.index[WINDOW - 1:][te_idx]
test_prices = df_main.loc[test_dates, "Close"]

# Trim to shortest
n_bt = min(len(test_prices), len(mm_preds))
test_prices = test_prices.iloc[:n_bt]
mm_signals  = mm_preds[:n_bt]

bt_results = run_full_backtest(
    prices   = test_prices,
    signals  = mm_signals,
    cfg      = cfg,
    ticker   = TICKER,
    save_dir = "../results",
)

print("\n── Strategy Metrics ──")
for k, v in bt_results["strategy"]["metrics"].items():
    print(f"  {k:>20} : {v:.4f}")

print("\n── Buy & Hold Metrics ──")
for k, v in bt_results["buy_and_hold"]["metrics"].items():
    print(f"  {k:>20} : {v:.4f}")

In [ ]:
# Display equity curve (already saved to results/)
from IPython.display import Image as IPImage
IPImage(f"../results/{TICKER}_equity_curve.png")

## 9 · Statistical Validation (Paired t-test)

In [ ]:
from src.evaluate import paired_t_test
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import accuracy_score

# Simulate fold-level accuracy scores for the t-test
# (use a quick 5-fold time-series split on the test set)
tss = TimeSeriesSplit(n_splits=5)
mm_fold_accs, bl_fold_accs = [], []

for _, fold_idx in tss.split(range(len(y_test))):
    mm_fold_accs.append(accuracy_score(y_test[fold_idx], mm_preds[fold_idx]))
    bl_fold_accs.append(accuracy_score(y_test[fold_idx], bl_preds[fold_idx]))

t_result = paired_t_test(
    np.array(bl_fold_accs),
    np.array(mm_fold_accs),
    metric_name="accuracy",
)
print(f"\n── Paired t-test result ──")
print(f"  H0: No significant difference between models")
print(f"  H1: Multi-modal significantly outperforms baseline")
print(f"  t-statistic : {t_result['t_statistic']:.4f}")
print(f"  p-value     : {t_result['p_value']:.4f}")
print(f"  Significant  (α=0.05): {t_result['significant']}")

## 10 · Summary Table

In [ ]:
summary = pd.DataFrame({
    "Metric":      ["Accuracy", "Precision", "Recall", "F1", "MCC", "AUC",
                    "ROI", "Sharpe Ratio", "Max Drawdown"],
    "Baseline LSTM": [
        bl_metrics.get("accuracy",  ""),
        bl_metrics.get("precision", ""),
        bl_metrics.get("recall",    ""),
        bl_metrics.get("f1",        ""),
        bl_metrics.get("mcc",       ""),
        bl_metrics.get("auc",       ""),
        bt_results["buy_and_hold"]["metrics"]["roi"],
        bt_results["buy_and_hold"]["metrics"]["sharpe_ratio"],
        bt_results["buy_and_hold"]["metrics"]["max_drawdown"],
    ],
    "Multi-Modal": [
        mm_metrics.get("accuracy",  ""),
        mm_metrics.get("precision", ""),
        mm_metrics.get("recall",    ""),
        mm_metrics.get("f1",        ""),
        mm_metrics.get("mcc",       ""),
        mm_metrics.get("auc",       ""),
        bt_results["strategy"]["metrics"]["roi"],
        bt_results["strategy"]["metrics"]["sharpe_ratio"],
        bt_results["strategy"]["metrics"]["max_drawdown"],
    ],
})

# Format floats
for col in ["Baseline LSTM", "Multi-Modal"]:
    summary[col] = summary[col].apply(lambda x: f"{x:.4f}" if isinstance(x, float) else x)

print(f"\n{'='*55}")
print(f"  {TICKER} — Final Results Summary")
print(f"{'='*55}")
print(summary.to_string(index=False))
summary.to_csv(f"../results/{TICKER}_summary.csv", index=False)
print(f"\nSaved → results/{TICKER}_summary.csv")

---
## Results are saved to `results/`

| File | Contents |
|------|----------|
| `{ticker}_mm_training.png`    | Multi-modal loss / accuracy curves |
| `{ticker}_bl_training.png`    | Baseline loss / accuracy curves |
| `{ticker}_mm_confusion.png`   | Multi-modal confusion matrix |
| `{ticker}_bl_confusion.png`   | Baseline confusion matrix |
| `{ticker}_mm_roc.png`         | Multi-modal ROC curves |
| `{ticker}_bl_roc.png`         | Baseline ROC curves |
| `{ticker}_benchmark.png`      | Benchmark comparison bar chart |
| `{ticker}_equity_curve.png`   | Equity + drawdown chart |
| `{ticker}_backtest_comparison.png` | Strategy vs B&H bar chart |
| `{ticker}_summary.csv`        | Final metrics table |